# Laser Off Code

In [ ]:
############################ TURN LASER OFF ############################ 
laser.enable(False)
print(f'Laser enable status: {laser.enable()}')

Investigating methodology for wavelength and vs attenuation sweep 

# Import

In [1]:
import time
from time import sleep, monotonic
import datetime
import numpy as np
import matplotlib.pyplot as plt
import sys
import pyvisa
import qcodes as qc
from qcodes.dataset import Measurement
from qcodes.dataset import do0d
from qcodes.dataset.experiment_container import new_experiment, load_experiment_by_name
from qcodes.dataset.plotting import plot_by_id
from qcodes.dataset.data_set import load_by_id, load_by_counter
from qcodes import initialise_or_create_database_at, new_data_set, new_experiment
from qcodes.station import Station
from ipywidgets import interact, IntSlider
initialise_or_create_database_at("./2026-05-11_SNSPD11.db")
import snspd
params = snspd.snspd('snspd11.yaml')

# Set up experiment
exp_name = 'SNSPD11_11_05_2026'
sample_name = '00'

try:
    exp = qc.load_experiment_by_name(exp_name, sample=sample_name)
    print('Experiment loaded. Last ID no:', exp.last_counter)
except ValueError:
    exp = new_experiment(exp_name, sample_name)
    print('Started new experiment')

Logging hadn't been started.
Activating auto-logging. Current session state plus future input saved.
Filename       : C:\Users\QNL\.qcodes\logs\command_history.log
Mode           : append
Output logging : True
Raw input log  : False
Timestamping   : True
State          : active
Qcodes Logfile : C:\Users\QNL\.qcodes\logs\260518-16308-qcodes.log
Experiment loaded. Last ID no: 143


In [2]:
import importlib
importlib.reload(snspd)
params = snspd.snspd('snspd11.yaml')

# Instruments

In [9]:
station = Station(config_file="friesland.yaml")
dmm = station.load_instrument("dmm", revive_instance=True)
yoko = station.load_instrument("yoko", revive_instance=True)
laser = station.load_instrument("laser", revive_instance=True)
MS = station.load_instrument("osc", revive_instance=True)
pm100d = station.load_instrument("pm100d", revive_instance=True) 
pms120 = station.load_instrument("pms120", revive_instance=True)
tc = station.load_instrument("fridge", revive_instance=True)
p_att = station.load_instrument("dmm_keithley", revive_instance=True) # excluding from snapshot because none of the parameters work anyway

Connected to: Thorlabs PM100D (serial:P0033329, firmware:2.8.1) in 5.53s


2026-05-18 15:04:37,243 ¦ qcodes.instrument.instrument_base ¦ WARNING ¦ instrument_base ¦ snapshot_base ¦ 464 ¦ [pm100d(Thorlabs_PM100D)] Snapshot: Could not update parameter: beam_diameter
2026-05-18 15:04:37,392 ¦ py.warnings ¦ WARNING ¦ warnings ¦ _showwarnmsg ¦ 110 ¦ C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\pyvisa\resources\messagebased.py:702: UserWarning: read string doesn't end with termination characters
  return self.read()



Connected to: Keithley instruments 2230G-30-1 (serial:9010428, firmware:1.16-1.04) in 0.15s


In [11]:
params.initialize_station()

In [13]:
params.att_blue_name

'V1550PA'

Check power stability while running the same code as wavelength sweep (device line 1). Arrangement is laser -> 10% port -> V1550PA -> pm100d and 90% port -> Pms120 (calibration arrangement) 

Aim is to see if results vary from the expected value. 

In [16]:
p_att.write('VOLT 5')
time.sleep(20)
print(f'Attenuator voltage: {p_att.ask('VOLT?')}')


l = [float(val) for val in params.laser_wav_range.values()]
laser_wav_range = np.arange(l[0], l[1]+l[2], l[2])

wavelength_range = laser_wav_range

##### same as counts vs wavelength function #####
for wav in wavelength_range: # <- sweep wavelength of laser 


    # Start with laser off 
    ############################ TURN LASER OFF ############################ 
    laser.enable(False)
    print(f'Laser enable status: {laser.enable()}')
    
    print(wav)
    
    # Set wavelength of laser and powermeters
    params.laser_set_standard(laser, wavelength=wav, power=7)
    params.laser_get_standard(laser)
    params.pmeter_set_standard(pmeter=params.pm100d, wavelength=wav)
    params.pmeter_set_standard(pmeter=params.pms120, wavelength=wav) # <- added for this measurement because 
    

    # ############################ TURN LASER ON ############################ 
    laser.enable(True)
    print(f'Laser enable status: {laser.enable()}')
    time.sleep(10)


    # Run calibration in place of counts measurement 
    params.calibrate(t=30, pmeter10=params.pm100d, pmeter90=params.pms120, attenuator_name=params.att_blue_name)
            

Attenuator voltage: 5
Laser enable status: False
1.528e-06
Power: 7.0
Frequency coarse: 196.1992THz
Wavelength (calculated) is 1528.0004097876035nm
Powermeter wavelength is 1.528e-06


2026-05-18 16:45:48,788 ¦ py.warnings ¦ WARNING ¦ warnings ¦ _showwarnmsg ¦ 110 ¦ C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\pyvisa\resources\messagebased.py:702: UserWarning: read string doesn't end with termination characters
  return self.read()



Powermeter wavelength is 1.528e-06
Laser enable status: True
update station
Starting experimental run with id: 144. 
144


2026-05-18 16:45:59,337 ¦ py.warnings ¦ WARNING ¦ warnings ¦ _showwarnmsg ¦ 110 ¦ C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\pyvisa\resources\messagebased.py:702: UserWarning: read string doesn't end with termination characters
  return self.read()



Finished in 43.94668609998189s


...

Finished in 43.94259990006685s
Laser enable status: False
1.5660000000000029e-06


2026-05-18 17:20:26,404 ¦ py.warnings ¦ WARNING ¦ warnings ¦ _showwarnmsg ¦ 110 ¦ C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\pyvisa\resources\messagebased.py:702: UserWarning: read string doesn't end with termination characters
  return self.read()



Power: 7.0
Frequency coarse: 191.4383THz
Wavelength (calculated) is 1566.0004189339331nm
Powermeter wavelength is 1.566e-06
Powermeter wavelength is 1.566e-06
Laser enable status: True
update station
Starting experimental run with id: 182. 
182


2026-05-18 17:20:36,965 ¦ py.warnings ¦ WARNING ¦ warnings ¦ _showwarnmsg ¦ 110 ¦ C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\pyvisa\resources\messagebased.py:702: UserWarning: read string doesn't end with termination characters
  return self.read()



Finished in 43.86794610018842s


Try attenuation sweep 

Code from ID 143

In [17]:
laser.enable()

True

In [20]:
############################ TURN LASER OFF ############################ 
laser.enable(False)
print(f'Laser enable status: {laser.enable()}')

params.laser_set_standard(laser, wavelength=1550e-9, power=7)
params.laser_get_standard(laser)
params.pmeter_set_standard(pmeter=pm100d, wavelength=1550e-9)
params.pmeter_set_standard(pmeter=pms120, wavelength=1550e-9)
# params.MSO5_set_standard_counts(MS) # <- comment out irrelevant lines 
# p_att.write(f'VOLT {v_att_range[-1]}') 
# yoko.current(-14e-6)


v = params.att_blue_v_attenuator_range
v_att_range = np.arange(v['start'], v['stop'], v['step'])
current = -14e-6

############################ TURN LASER ON ############################ 
laser.enable(True)
print(f'Laser enable status: {laser.enable()}')
time.sleep(10)


### Insert relevant parts of MSO5_counts_vs_attenuation ###

# Set attenuator to max value
p_att.write(f'VOLT {v_att_range[-1]}')
time.sleep(5)

# # Set current
# yoko.current(current)
# time.sleep(1)
# print(f'Current is {yoko.current()}')

if MS.channels[0].clipping(): 
    print('Error: Clipping')

# Extract the amount of time in one trace 
h_time = MS.horizontal_scale()*MS.horizontal_divisions()

for v in v_att_range[::-1]: # <- sweep voltage applied to attenuator
    
    p_att.write(f'VOLT {v}')
    time.sleep(1)
    print(f'Starting V={p_att.ask('VOLT?')}')

    #### Run calibration in place of counts measurement ###
    params.calibrate(t=30, pmeter10=params.pm100d, pmeter90=params.pms120, attenuator_name=params.att_blue_name)

############################ TURN LASER OFF ############################ 
laser.enable(False)
print(f'Laser enable status: {laser.enable()}')

2026-05-19 10:17:06,628 ¦ py.warnings ¦ WARNING ¦ warnings ¦ _showwarnmsg ¦ 110 ¦ C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\pyvisa\resources\messagebased.py:702: UserWarning: read string doesn't end with termination characters
  return self.read()



Laser enable status: False
Power: 7.0
Frequency coarse: 193.4144THz
Wavelength (calculated) is 1550.000713493928nm
Powermeter wavelength is 1.55e-06
Powermeter wavelength is 1.55e-06
Laser enable status: True
Current is -1.4e-05
Starting V=5.5
update station
Starting experimental run with id: 183. 
183


2026-05-19 10:17:24,190 ¦ py.warnings ¦ WARNING ¦ warnings ¦ _showwarnmsg ¦ 110 ¦ C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\pyvisa\resources\messagebased.py:702: UserWarning: read string doesn't end with termination characters
  return self.read()



Finished in 43.941157599911094s
Starting V=5.45
update station
Starting experimental run with id: 184. 
184


2026-05-19 10:18:09,605 ¦ py.warnings ¦ WARNING ¦ warnings ¦ _showwarnmsg ¦ 110 ¦ C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\pyvisa\resources\messagebased.py:702: UserWarning: read string doesn't end with termination characters
  return self.read()



...

Finished in 43.93868500017561s
Starting V=3.45
update station
Starting experimental run with id: 224. 
224


2026-05-19 10:48:26,453 ¦ py.warnings ¦ WARNING ¦ warnings ¦ _showwarnmsg ¦ 110 ¦ C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\pyvisa\resources\messagebased.py:702: UserWarning: read string doesn't end with termination characters
  return self.read()



Finished in 43.921689399983734s
Starting V=3.4
update station
Starting experimental run with id: 225. 
225


2026-05-19 10:49:11,821 ¦ py.warnings ¦ WARNING ¦ warnings ¦ _showwarnmsg ¦ 110 ¦ C:\Users\QNL\anaconda3\envs\qcodes\Lib\site-packages\pyvisa\resources\messagebased.py:702: UserWarning: read string doesn't end with termination characters
  return self.read()



Finished in 43.85829110001214s
Laser enable status: False


laser.